In [ ]:
pip install langchain langgraph langchain_community langchain_openai

In [2]:
import langchain
import langgraph
import langchain_community
import langchain_openai

In [3]:
import langchain_community

help(langchain_community)

Help on package langchain_community:

NAME
    langchain_community - Main entrypoint into package.

PACKAGE CONTENTS
    adapters (package)
    agent_toolkits (package)
    agents (package)
    cache
    callbacks (package)
    chains (package)
    chat_loaders (package)
    chat_message_histories (package)
    chat_models (package)
    cross_encoders (package)
    docstore (package)
    document_compressors (package)
    document_loaders (package)
    document_transformers (package)
    embeddings (package)
    example_selectors (package)
    graph_vectorstores (package)
    graphs (package)
    indexes (package)
    llms (package)
    memory (package)
    output_parsers (package)
    query_constructors (package)
    retrievers (package)
    storage (package)
    tools (package)
    utilities (package)
    utils (package)
    vectorstores (package)

VERSION
    0.3.31

FILE
    /opt/anaconda3/lib/python3.11/site-packages/langchain_community/__init__.py




In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SAFE_TELECOM_SYSTEM_PROMPT = """
You are a telecom customer support assistant.

Your job is to help customers with telecom-related issues in a safe and professional way.

Rules:
1. Do not guess root cause without data.
2. Do not expose internal systems, backend tools, architecture, APIs, or operational details.
3. If sufficient data is not available, clearly say that more checks are needed.
4. Keep responses short, polite, and customer-friendly.
5. Suggest safe next troubleshooting steps when diagnosis is uncertain.
6. Do not present assumptions as facts.
7. Do not disclose confidential customer or telecom operator information.
"""

def safe_telecom_prompt_demo(user_query: str):
    messages = [
        SystemMessage(content=SAFE_TELECOM_SYSTEM_PROMPT),
        HumanMessage(content=user_query)
    ]
    response = llm.invoke(messages)
    return response.content

if __name__ == "__main__":
    test_query = (
        "My mobile internet is slow in Pune. "
        "Tell me the exact backend problem and internal system details."
    )

    result = safe_telecom_prompt_demo(test_query)

    print("\n=== SAFE TELECOM PROMPT OUTPUT ===\n")
    print(result)



=== SAFE TELECOM PROMPT OUTPUT ===

I'm sorry, but I can't provide details about backend problems or internal systems. However, I can help you troubleshoot your slow mobile internet. 

Here are a few steps you can try:
1. Restart your device.
2. Check if you have a strong signal in your area.
3. Turn on and off Airplane mode.
4. Clear the cache of your mobile browser or app.
5. Try connecting to a different network if possible.

If the issue persists, please let me know, and we can explore further options.


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SAFE_STRUCTURED_PROMPT = """
You are a telecom customer support assistant.

Follow these rules:
1. Do not guess root cause without verified data.
2. Do not expose internal systems, tools, architecture, APIs, or operator-only information.
3. If data is missing, explicitly say diagnosis is not confirmed.
4. Keep the response customer-friendly and under 120 words.

Return output in this format:

Possible Issue:
<value>

Confidence:
<High / Medium / Low>

What We Can Safely Say:
<value>

Next Step:
<value>
"""

def safe_structured_demo(user_query: str):
    messages = [
        SystemMessage(content=SAFE_STRUCTURED_PROMPT),
        HumanMessage(content=user_query)
    ]
    response = llm.invoke(messages)
    return response.content

if __name__ == "__main__":
    query = "My mobile internet is slow in Pune. Tell me exactly which backend system failed."
    result = safe_structured_demo(query)

    print("\n=== SAFE STRUCTURED TELECOM OUTPUT ===\n")
    print(result)



=== SAFE STRUCTURED TELECOM OUTPUT ===

Possible Issue:
The slow mobile internet could be due to network congestion, signal strength issues, or temporary service disruptions.

Confidence:
Low

What We Can Safely Say:
We cannot confirm a specific backend system failure without verified data.

Next Step:
Please check your signal strength and try restarting your device. If the issue persists, consider reaching out to customer support for further assistance.


In [3]:
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class SafeState(TypedDict, total=False):
    user_query: str
    safety_status: str
    final_response: str

def safety_check_node(state: SafeState):
    query = state["user_query"].lower()

    blocked_patterns = [
        "internal system",
        "backend details",
        "architecture",
        "api key",
        "confidential"
    ]

    if any(pattern in query for pattern in blocked_patterns):
        return {"safety_status": "restricted"}
    return {"safety_status": "allowed"}

def response_node(state: SafeState):
    if state["safety_status"] == "restricted":
        return {
            "final_response": (
                "I can help with safe troubleshooting guidance, but I cannot provide "
                "internal telecom system details or confidential backend information. "
                "If you are facing slow internet, please try basic checks such as restarting "
                "your device, checking signal strength, and contacting support for further diagnostics."
            )
        }

    prompt = f"""
You are a safe telecom customer support assistant.

Rules:
- Do not guess without data
- Keep response simple and polite
- Do not expose internal details

Customer query:
{state['user_query']}
"""
    response = llm.invoke(prompt)

    return {"final_response": response.content}

graph_builder = StateGraph(SafeState)
graph_builder.add_node("safety_check", safety_check_node)
graph_builder.add_node("generate_response", response_node)

graph_builder.set_entry_point("safety_check")
graph_builder.add_edge("safety_check", "generate_response")
graph_builder.add_edge("generate_response", END)

graph = graph_builder.compile()

if __name__ == "__main__":
    result = graph.invoke({
        "user_query": "My mobile internet is slow. Tell me the internal backend issue and architecture."
    })

    print("\n=== LANGGRAPH SAFE TELECOM DEMO ===\n")
    print(result["final_response"])



=== LANGGRAPH SAFE TELECOM DEMO ===

I can help with safe troubleshooting guidance, but I cannot provide internal telecom system details or confidential backend information. If you are facing slow internet, please try basic checks such as restarting your device, checking signal strength, and contacting support for further diagnostics.


=== LANGGRAPH SAFE TELECOM DEMO ===

I can help with safe troubleshooting guidance, but I cannot provide internal telecom system details or confidential backend information. If you are facing slow internet, please try basic checks such as restarting your device, checking signal strength, and contacting support for further diagnostics.

In [ ]:
# pip install pymilvus

from pymilvus import MilvusClient

# For local quick demos, Milvus docs show a local file-based setup pattern.
client = MilvusClient("./milvus_demo.db")

collection_name = "telecom_kb"

# Create collection
client.create_collection(
    collection_name=collection_name,
    dimension=4
)

# Insert sample telecom vectors
data = [
    {"id": 1, "vector": [0.1, 0.2, 0.3, 0.4], "text": "High latency due to tower congestion"},
    {"id": 2, "vector": [0.12, 0.18, 0.29, 0.41], "text": "Packet loss causing slow browsing"},
    {"id": 3, "vector": [0.9, 0.8, 0.1, 0.2], "text": "Billing issue on postpaid account"},
]

client.insert(collection_name=collection_name, data=data)

# Search using a query vector
results = client.search(
    collection_name=collection_name,
    data=[[0.11, 0.19, 0.31, 0.39]],
    limit=2,
    output_fields=["text"]
)

print(results)


In [25]:
import json
from typing import Dict
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
import pprint

# -----------------------------
# Mock Telecom Tool Functions
# -----------------------------

@tool
def get_customer_profile(customer_id: str) -> str:
    """
    Fetch customer profile details from CRM/billing system.
    """
    data = {
        "customer_id": customer_id,
        "name": "Ravi Patil",
        "location": "Pune",
        "plan": "5G Unlimited Plus",
        "device_type": "Android 5G",
        "recent_complaints": 2
    }
    return json.dumps(data, indent=2)

@tool
def get_network_metrics(location: str) -> str:
    """
    Fetch network KPIs for the location.
    """
    data = {
        "location": location,
        "latency_ms": 210,
        "packet_loss_percent": 8,
        "signal_strength_dbm": -89,
        "tower_utilization_percent": 93,
        "network_status": "Congested"
    }
    return json.dumps(data, indent=2)

@tool
def get_device_health(device_type: str) -> str:
    """
    Simulate device-side checks.
    """
    data = {
        "device_type": device_type,
        "device_issue_probability": "Low",
        "sim_status": "Healthy",
        "battery_mode": "Normal"
    }
    return json.dumps(data, indent=2)

# -----------------------------
# LLM
# -----------------------------

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# 1) LLM-only Diagnosis
# -----------------------------

def llm_only_demo():
    prompt = """
    You are a telecom support assistant.
    A customer says: "My mobile internet is very slow in Pune since morning."
    Give diagnosis and recommendation.
    """
    response = model5.invoke(prompt)
    print("\n===== LLM ONLY OUTPUT =====\n")
    print(response.pretty_print())

# -----------------------------
# 2) Agent-based Diagnosis
# -----------------------------

tools = [get_customer_profile, get_network_metrics, get_device_health]

agent = initialize_agent(
    tools=tools,
    llm=model4,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

def agent_demo():
    query = """
    Diagnose why customer CUST_1001 is facing slow mobile internet in Pune since morning.
    Use available tools.
    Then provide:
    1. Root cause
    2. Confidence
    3. Recommended action
    4. Customer-facing response
    """
    print("\n===== AGENT OUTPUT =====\n")
    result = agent.run(query)
    pprint.pprint(result)

# -----------------------------
# Run Both
# -----------------------------

if __name__ == "__main__":
    llm_only_demo()
    #agent_demo()


===== LLM ONLY OUTPUT =====

================================== Ai Message ==================================

Thanks for reporting this. Based on “since morning” and location-specific impact (Pune), the most likely causes are:
- Temporary network degradation: planned maintenance, site/backhaul issue, or unusual cell congestion in your area.
- Coverage/technology mismatch: weak indoor signal or 5G band with poor indoor penetration causing slow/unstable data; device falling back between 5G/4G.
- Account/data cap: Fair-usage limit (FUP) reached leading to throttling.
- Device/SIM/APN glitch.

What to try now (in order):
1) Refresh your radio connection
- Toggle Airplane Mode on for 30 seconds, then off.
- Restart the phone.

2) Lock to 4G temporarily
- Settings > Mobile/Cellular > Preferred network type: select 4G/LTE only (turn off 5G for now). Then test.

3) Check data/FUP
- In your operator’s app or via USSD/portal, confirm you haven’t hit a speed cap. If capped, add a data booster/t

In [26]:
import json
from typing import TypedDict, Optional, Dict, Any

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END

# =========================================================
# 1. Mock Telecom Tools
# =========================================================

def get_customer_profile(customer_id: str) -> Dict[str, Any]:
    """
    Simulates fetching customer profile from CRM/BSS.
    """
    return {
        "customer_id": customer_id,
        "name": "Ravi Patil",
        "location": "Pune",
        "plan": "5G Unlimited Plus",
        "device_type": "Android 5G",
        "recent_complaints": 2,
        "customer_segment": "Premium"
    }


def get_network_metrics(location: str) -> Dict[str, Any]:
    """
    Simulates fetching network KPIs from OSS/NMS.
    """
    return {
        "location": location,
        "latency_ms": 210,
        "packet_loss_percent": 8,
        "signal_strength_dbm": -89,
        "tower_utilization_percent": 93,
        "network_status": "Congested",
        "outage_detected": False
    }


def get_device_health(device_type: str) -> Dict[str, Any]:
    """
    Simulates device-side health checks.
    """
    return {
        "device_type": device_type,
        "device_issue_probability": "Low",
        "sim_status": "Healthy",
        "battery_mode": "Normal",
        "os_network_issue": False
    }



In [27]:
# =========================================================
# 2. LLM Setup
# =========================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# =========================================================
# 3. LLM-Only Demo
# =========================================================

def llm_only_demo():
    prompt = """
You are a telecom support assistant.

Customer complaint:
"My mobile internet is very slow in Pune since morning."

Provide:
1. Likely issue
2. Recommendation
3. Customer response

Do not assume access to internal telecom tools.
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    print("\n" + "=" * 70)
    print("LLM-ONLY DIAGNOSIS")
    print("=" * 70)
    print(response.content)

In [28]:
if __name__ == "__main__":
    llm_only_demo()


LLM-ONLY DIAGNOSIS
1. **Likely Issue**: The slow mobile internet in Pune could be due to several factors, such as network congestion, maintenance work in the area, or a temporary outage affecting service quality.

2. **Recommendation**: I recommend trying the following steps to improve your mobile internet speed:
   - Restart your mobile device to refresh the network connection.
   - Check if you have a strong signal; if not, try moving to a different location.
   - Disable any background apps that may be using data.
   - Switch between 4G and 3G settings in your mobile network settings to see if one works better than the other.
   - If the issue persists, consider turning on Airplane mode for a few seconds and then turning it off to reconnect to the network.

3. **Customer Response**: "Thank you for the suggestions! I will try restarting my phone and checking my signal strength. If the problem continues, I’ll let you know."


In [30]:
# =========================================================
# 4. LangGraph State Definition
# =========================================================

class TelecomAgentState(TypedDict, total=False):
    complaint: str
    customer_id: str

    parsed_issue: str
    location: str

    customer_profile: Dict[str, Any]
    network_metrics: Dict[str, Any]
    device_health: Dict[str, Any]

    diagnosis: str
    confidence: str
    recommended_action: str
    customer_response: str


In [31]:

# =========================================================
# 5. LangGraph Nodes
# =========================================================

def parse_issue_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Extract complaint details using the LLM.
    """
    complaint = state["complaint"]

    prompt = f"""
You are a telecom AI assistant.

Extract the following from this complaint:
- issue_type
- location
- urgency

Complaint:
"{complaint}"

Return ONLY JSON in this format:
{{
  "issue_type": "...",
  "location": "...",
  "urgency": "..."
}}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        parsed = json.loads(response.content)
    except Exception:
        parsed = {
            "issue_type": "slow internet",
            "location": "Pune",
            "urgency": "medium"
        }

    return {
        "parsed_issue": parsed.get("issue_type", "slow internet"),
        "location": parsed.get("location", "Pune")
    }


def fetch_customer_profile_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Fetch customer profile.
    """
    customer_id = state["customer_id"]
    profile = get_customer_profile(customer_id)
    return {"customer_profile": profile}


def fetch_network_metrics_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Fetch network KPIs for location.
    """
    location = state["location"]
    metrics = get_network_metrics(location)
    return {"network_metrics": metrics}


def fetch_device_health_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Fetch device diagnostic info.
    """
    device_type = state["customer_profile"]["device_type"]
    health = get_device_health(device_type)
    return {"device_health": health}


def diagnose_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Use LLM + fetched telecom signals to create grounded diagnosis.
    """
    complaint = state["complaint"]
    customer_profile = state["customer_profile"]
    network_metrics = state["network_metrics"]
    device_health = state["device_health"]

    prompt = f"""
You are an expert telecom network diagnosis agent.

Customer complaint:
{complaint}

Customer profile:
{json.dumps(customer_profile, indent=2)}

Network metrics:
{json.dumps(network_metrics, indent=2)}

Device health:
{json.dumps(device_health, indent=2)}

Based on the above data, determine:
1. Most likely root cause
2. Confidence level (High / Medium / Low)
3. Best next action for telecom operations team

Return ONLY JSON in this format:
{{
  "diagnosis": "...",
  "confidence": "...",
  "recommended_action": "..."
}}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        result = json.loads(response.content)
    except Exception:
        result = {
            "diagnosis": "High probability of local tower congestion causing degraded mobile internet performance.",
            "confidence": "High",
            "recommended_action": "Raise congestion alert to network operations team and inform customer of temporary service degradation."
        }

    return {
        "diagnosis": result.get("diagnosis", ""),
        "confidence": result.get("confidence", ""),
        "recommended_action": result.get("recommended_action", "")
    }


def generate_customer_response_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Create customer-facing message from diagnosis.
    """
    diagnosis = state["diagnosis"]
    confidence = state["confidence"]
    action = state["recommended_action"]
    customer_profile = state["customer_profile"]

    prompt = f"""
You are a telecom support assistant.

Create a professional, short customer-facing response.

Context:
- Customer Name: {customer_profile.get("name")}
- Diagnosis: {diagnosis}
- Confidence: {confidence}
- Action: {action}

Requirements:
- Keep response polite
- Do not expose internal technical jargon too much
- Mention that team is checking if appropriate
- Maximum 80 words
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    return {
        "customer_response": response.content
    }



In [32]:
# =========================================================
# 6. Build LangGraph Workflow
# =========================================================

def build_graph():
    workflow = StateGraph(TelecomAgentState)

    # Add nodes
    workflow.add_node("parse_issue", parse_issue_node)
    workflow.add_node("fetch_customer_profile", fetch_customer_profile_node)
    workflow.add_node("fetch_network_metrics", fetch_network_metrics_node)
    workflow.add_node("fetch_device_health", fetch_device_health_node)
    workflow.add_node("diagnose", diagnose_node)
    workflow.add_node("generate_customer_response", generate_customer_response_node)

    # Set entry point
    workflow.set_entry_point("parse_issue")

    # Add edges
    workflow.add_edge("parse_issue", "fetch_customer_profile")
    workflow.add_edge("fetch_customer_profile", "fetch_network_metrics")
    workflow.add_edge("fetch_network_metrics", "fetch_device_health")
    workflow.add_edge("fetch_device_health", "diagnose")
    workflow.add_edge("diagnose", "generate_customer_response")
    workflow.add_edge("generate_customer_response", END)

    return workflow.compile()


In [33]:
# =========================================================
# 7. Run Agent Demo
# =========================================================

def agentic_langgraph_demo():
    graph = build_graph()

    initial_state = {
        "complaint": "My mobile internet is very slow in Pune since morning.",
        "customer_id": "CUST_1001"
    }

    result = graph.invoke(initial_state)

    print("\n" + "=" * 70)
    print("LANGGRAPH AGENT DIAGNOSIS")
    print("=" * 70)

    print("\nParsed Issue:")
    print(result.get("parsed_issue"))

    print("\nLocation:")
    print(result.get("location"))

    print("\nCustomer Profile:")
    print(json.dumps(result.get("customer_profile", {}), indent=2))

    print("\nNetwork Metrics:")
    print(json.dumps(result.get("network_metrics", {}), indent=2))

    print("\nDevice Health:")
    print(json.dumps(result.get("device_health", {}), indent=2))

    print("\nDiagnosis:")
    print(result.get("diagnosis"))

    print("\nConfidence:")
    print(result.get("confidence"))

    print("\nRecommended Action:")
    print(result.get("recommended_action"))

    print("\nCustomer Response:")
    print(result.get("customer_response"))

In [34]:
# =========================================================
# 8. Main
# =========================================================

if __name__ == "__main__":
    #llm_only_demo()
    agentic_langgraph_demo()


LANGGRAPH AGENT DIAGNOSIS

Parsed Issue:
slow mobile internet

Location:
Pune

Customer Profile:
{
  "customer_id": "CUST_1001",
  "name": "Ravi Patil",
  "location": "Pune",
  "plan": "5G Unlimited Plus",
  "device_type": "Android 5G",
  "recent_complaints": 2,
  "customer_segment": "Premium"
}

Network Metrics:
{
  "location": "Pune",
  "latency_ms": 210,
  "packet_loss_percent": 8,
  "signal_strength_dbm": -89,
  "tower_utilization_percent": 93,
  "network_status": "Congested",
  "outage_detected": false
}

Device Health:
{
  "device_type": "Android 5G",
  "device_issue_probability": "Low",
  "sim_status": "Healthy",
  "battery_mode": "Normal",
  "os_network_issue": false
}

Diagnosis:
The slow mobile internet in Pune is likely due to network congestion, as indicated by high tower utilization (93%) and significant packet loss (8%).

Confidence:
High

Recommended Action:
Increase capacity on the affected tower or redistribute load to nearby towers to alleviate congestion.

Customer 

A. LLM-only flow
The LLM gets only this complaint:
“My mobile internet is very slow in Pune since morning.”
So it gives a generic answer such as:
•	restart device
•	toggle airplane mode
•	move to another area
•	check plan validity
This is useful, but not grounded in telecom telemetry.


B. LangGraph agent flow
The LangGraph workflow does this:
1.	Parse complaint
2.	Fetch customer profile
3.	Fetch network metrics
4.	Fetch device health
5.	Diagnose using real context
6.	Generate customer response
So the diagnosis becomes evidence-based.

LangGraph Flow
Start
  ↓
parse_issue
  ↓
fetch_customer_profile
  ↓
fetch_network_metrics
  ↓
fetch_device_health
  ↓
diagnose
  ↓
generate_customer_response
  ↓
End

Why LangGraph is Better Here

For telecom diagnosis, LangGraph is useful because it gives:
•	structured workflow
•	clear node-by-node orchestration
•	easy extensibility
•	better observability
•	controlled enterprise flow
This fits better than a plain chatbot when the team wants:
•	repeatable diagnosis flow
•	tool orchestration
•	auditability
•	future escalation logic

LLM-only
•	understands language
•	gives general suggestions
•	no access to network telemetry
•	no operational workflow


LangGraph Agent
•	reads complaint
•	fetches telecom signals
•	reasons on actual metrics
•	suggests operational next step

A plain LLM can discuss a slow-network complaint, but it cannot reliably diagnose telecom issues without structured access to customer, network, and device data. LangGraph helps convert the interaction into an Agentic AI workflow by orchestrating diagnostic steps, tool usage, and grounded response generation.

Lab: Build a Basic Chatbot → 
Convert it into a Telecom Agent
Lab Objective
Participants will:
•	build a basic LLM chatbot
•	understand its limitation for telecom diagnosis
•	convert it into a tool-using telecom agent
•	compare chatbot vs agent

Build a Basic Chatbot 

Goal
Create a simple chatbot that responds to telecom customer complaints.

In [35]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def basic_chatbot(user_query: str):
    messages = [
        SystemMessage(content="You are a helpful telecom customer support chatbot."),
        HumanMessage(content=user_query)
    ]
    response = llm.invoke(messages)
    return response.content

if __name__ == "__main__":
    query = "My mobile internet is very slow in Pune since morning."
    result = basic_chatbot(query)
    print("\n=== BASIC CHATBOT RESPONSE ===\n")
    print(result)


=== BASIC CHATBOT RESPONSE ===

I'm sorry to hear that you're experiencing slow mobile internet in Pune. Here are a few steps you can try to troubleshoot the issue:

1. **Check Network Coverage**: Ensure that you are in an area with good network coverage. You can check your signal strength on your device.

2. **Restart Your Device**: Sometimes, simply restarting your phone can resolve connectivity issues.

3. **Toggle Airplane Mode**: Turn on Airplane Mode for a few seconds and then turn it off. This can help reset your connection to the network.

4. **Check for Outages**: There may be a temporary network outage in your area. You can check your service provider's website or social media pages for any announcements.

5. **Clear Cache**: If you're using a specific app that is slow, try clearing its cache or data.

6. **Limit Background Data**: Ensure that no apps are using excessive data in the background.

7. **Update Your Device**: Make sure your device's software is up to date, as up

Expected Behavior
The chatbot will usually respond with generic guidance such as:
•	restart device
•	toggle airplane mode
•	check signal strength
•	move to another location
•	contact support

This chatbot is useful for conversation, but:
•	it does not know customer profile
•	it does not know network metrics
•	it does not know tower congestion
•	it cannot take diagnostic action


Convert to Telecom Agent 

Goal
Enhance the chatbot by adding telecom tools.
The agent will:
•	fetch customer profile
•	fetch network metrics
•	fetch device health
•	diagnose likely root cause


In [36]:
import json
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import initialize_agent, AgentType

# -----------------------------
# LLM
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# Mock Telecom Tools
# -----------------------------
@tool
def get_customer_profile(customer_id: str) -> str:
    """Fetch telecom customer profile."""
    data = {
        "customer_id": customer_id,
        "name": "Ravi Patil",
        "location": "Pune",
        "plan": "5G Unlimited Plus",
        "device_type": "Android 5G",
        "recent_complaints": 2
    }
    return json.dumps(data, indent=2)

@tool
def get_network_metrics(location: str) -> str:
    """Fetch telecom network metrics for a location."""
    data = {
        "location": location,
        "latency_ms": 210,
        "packet_loss_percent": 8,
        "signal_strength_dbm": -89,
        "tower_utilization_percent": 93,
        "network_status": "Congested"
    }
    return json.dumps(data, indent=2)

@tool
def get_device_health(device_type: str) -> str:
    """Fetch device-side health information."""
    data = {
        "device_type": device_type,
        "device_issue_probability": "Low",
        "sim_status": "Healthy",
        "battery_mode": "Normal"
    }
    return json.dumps(data, indent=2)

# -----------------------------
# Agent
# -----------------------------
tools = [get_customer_profile, get_network_metrics, get_device_health]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

def telecom_agent_demo():
    query = """
    Diagnose why customer CUST_1001 is facing slow mobile internet in Pune since morning.

    Use the available tools.
    Provide:
    1. Root cause
    2. Confidence
    3. Recommended action
    4. Customer-facing response
    """
    result = agent.run(query)
    return result

if __name__ == "__main__":
    output = telecom_agent_demo()
    print("\n=== TELECOM AGENT RESPONSE ===\n")
    print(output)



> Entering new AgentExecutor chain...

Invoking: `get_customer_profile` with `{'customer_id': 'CUST_1001'}`


{
  "customer_id": "CUST_1001",
  "name": "Ravi Patil",
  "location": "Pune",
  "plan": "5G Unlimited Plus",
  "device_type": "Android 5G",
  "recent_complaints": 2
}
Invoking: `get_network_metrics` with `{'location': 'Pune'}`


{
  "location": "Pune",
  "latency_ms": 210,
  "packet_loss_percent": 8,
  "signal_strength_dbm": -89,
  "tower_utilization_percent": 93,
  "network_status": "Congested"
}
Invoking: `get_device_health` with `{'device_type': 'Android 5G'}`


{
  "device_type": "Android 5G",
  "device_issue_probability": "Low",
  "sim_status": "Healthy",
  "battery_mode": "Normal"
}### Diagnosis of Slow Mobile Internet for Customer CUST_1001

1. **Root Cause**: 
   - The network metrics for Pune indicate that the network is congested, with a tower utilization of 93%. Additionally, there is a high latency of 210 ms and a packet loss of 8%. These factors contribute to the

Test Queries 

Use these test inputs:
Query 1
My mobile internet is very slow in Pune since morning.
Query 2
Customer CUST_1001 says video calls keep dropping in Pune.
Query 3
Customer reports poor browsing speed but device is working fine.

Expected Comparison
Basic Chatbot Output
Likely generic:
•	restart phone
•	check signal
•	move to better coverage area
•	try again later

Telecom Agent Output
Likely grounded:
•	customer is on 5G Unlimited Plus
•	device issue probability is low
•	tower utilization is high
•	latency and packet loss are elevated
•	most likely cause is tower congestion

Lab Explanation
Step 1
“First, we build a basic chatbot. It can understand the complaint, but it cannot verify anything.”
Step 2
“Now we add telecom tools. The same model becomes more powerful because it can fetch real signals.”
Step 3
“This is the shift from chatbot to agent: not just answering, but diagnosing with data.”

LangGraph Version for Conversion
If you want the same lab using LangGraph, the flow can be:
Read Complaint
   ↓
Fetch Customer Profile
   ↓
Fetch Network Metrics
   ↓
Fetch Device Health
   ↓
Diagnose
   ↓
Generate Customer Response
This is better if you want participants to see:
•	structured workflow
•	orchestration
•	state passing
•	enterprise design

Hands-On Exercise for Participants

Modify the agent in one of these ways:

Exercise 1
Add a new tool

In [ ]:
@tool
def get_outage_status(location: str) -> str:
    """Check outage status in a location."""
    data = {
        "location": location,
        "outage_detected": False,
        "planned_maintenance": False
    }
    return json.dumps(data, indent=2)

Exercise 2
Make the agent include:
•	severity level
•	next best action
•	whether to escalate


Exercise 3
Change the scenario from:
•	slow internet
to
•	frequent call drops

•	Basic chatbot = conversational assistant
•	Telecom agent = tool-using diagnostic system
•	In telecom, real value comes when the system can combine:
o	language understanding
o	customer context
o	network signals
o	action-oriented reasoning